In [1]:
import pandas as pd
import sys
!{sys.executable} -m pip install requests
from api_requests import baixar_e_descompactar


[notice] A new release of pip available: 22.3 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [2]:
from api_requests import baixar_e_descompactar

api_url = [
    'https://dadosabertos.ans.gov.br/FTP/PDA/demonstracoes_contabeis/2025/1T2025.zip',
    'https://dadosabertos.ans.gov.br/FTP/PDA/demonstracoes_contabeis/2025/2T2025.zip',
    'https://dadosabertos.ans.gov.br/FTP/PDA/demonstracoes_contabeis/2025/3T2025.zip',
    'https://dadosabertos.ans.gov.br/FTP/PDA/operadoras_acreditadas/operadoras_acreditadas.csv',
    'https://dadosabertos.ans.gov.br/FTP/PDA/operadoras_de_plano_de_saude_ativas/Relatorio_cadop.csv',
    'https://dadosabertos.ans.gov.br/FTP/PDA/operadoras_de_plano_de_saude_canceladas/Relatorio_cadop_canceladas.csv',
    'https://dadosabertos.ans.gov.br/FTP/PDA/operadoras_e_prestadores_nao_hospitalares/operadoras_e_prestadores_nao_hospitalares.zip'
]

baixar_e_descompactar(api_url)


Baixando 1T2025.zip...
Descompactando 1T2025.zip
Baixando 2T2025.zip...
Descompactando 2T2025.zip
Baixando 3T2025.zip...
Descompactando 3T2025.zip
Baixando operadoras_acreditadas.csv...
Baixando Relatorio_cadop.csv...
Baixando Relatorio_cadop_canceladas.csv...
Baixando operadoras_e_prestadores_nao_hospitalares.zip...
Descompactando operadoras_e_prestadores_nao_hospitalares.zip


In [28]:
import pandas as pd
import os

#apenas arquivos dos trimestres
paths = [
    os.path.join("data", f)
    for f in os.listdir("data")
    if f.endswith(".csv") and f.startswith(("1T", "2T", "3T"))
]

#next para o primeiro chunk
def le_exibe_colunas(path):
    chunk = next(pd.read_csv(path, sep=';', encoding='latin1', chunksize=100_000))
    print(chunk.columns.tolist())
    
    
for path in paths:
    print(f"\nArquivo: {path}")
    le_exibe_colunas(path)

def trimestre_filtrados(path, valor, coluna='DESCRICAO'):
    for chunk in pd.read_csv(path, sep=';', encoding='latin1', chunksize=100_000):
        mask = (
            chunk[coluna]
            .astype(str)
            .str.strip()
            .str.upper()
            .str.contains(valor.upper(), na=False, regex=True)
        )
        if mask.any():
            yield chunk.loc[mask]
            
df_eventos = pd.concat(
    (
        chunk
        for path in paths
        for chunk in trimestre_filtrados(path, r'(?=.*EVENT)(?=.*SINISTR)')
    ),
    ignore_index=True
)

data = os.path.join('data', 'eventos_sinistros_2025.csv')

df_eventos.to_csv(data, sep=';', index=False)
 
    



Arquivo: data/1T2025.csv
['DATA', 'REG_ANS', 'CD_CONTA_CONTABIL', 'DESCRICAO', 'VL_SALDO_INICIAL', 'VL_SALDO_FINAL']

Arquivo: data/2T2025.csv
['DATA', 'REG_ANS', 'CD_CONTA_CONTABIL', 'DESCRICAO', 'VL_SALDO_INICIAL', 'VL_SALDO_FINAL']

Arquivo: data/3T2025.csv
['DATA', 'REG_ANS', 'CD_CONTA_CONTABIL', 'DESCRICAO', 'VL_SALDO_INICIAL', 'VL_SALDO_FINAL']
